In [1]:
from pathlib import Path

import polars as pl

from krasnal.config import MOVE_VOCAB_PATH
from krasnal.dataset import PretrainDataset
from krasnal.tokens import (
    GAME_END_ID,
    GAME_START_ID,
    ID_TO_MOVE,
    load_move_vocab,
)

load_move_vocab(
    Path("..") / MOVE_VOCAB_PATH,
    piece_aware_moves=True,
    side_prefixed_moves=True,
)

FILTERED_DIR = Path("../data/1_filtered")
TOKENIZED_DIR = Path("../data/2_tokenized")

/home/igorjakus/Projects/bachelor-thesis/krasnal/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Stage 1: Filtered Games (UCI + evals + FEN)


In [2]:
# Load filtered games
filtered_files = sorted(FILTERED_DIR.glob("*.parquet"))[:3]
print("Filtered game files:", len(list(FILTERED_DIR.glob("*.parquet"))))
print("Sample files:", [f.name for f in filtered_files])

Filtered game files: 5
Sample files: ['filtered_2017-04.parquet', 'filtered_2017-05.parquet', 'filtered_2017-06.parquet']


In [3]:
# Load sample filtered game
filtered_sample = pl.read_parquet(filtered_files[0], n_rows=5)
print("Columns:", filtered_sample.columns)
print("\nSample row:")
filtered_sample

Columns: ['lichess_id', 'uci_moves', 'clocks_white', 'clocks_black', 'evals_cp', 'evals_raw', 'is_check', 'is_capture', 'piece_moved', 'promotion', 'is_en_passant', 'white_rating', 'black_rating', 'result', 'game_end_reason', 'time_initial', 'time_increment', 'utc_timestamp', 'opening', 'eco', 'ply_count', 'fen']

Sample row:


lichess_id,uci_moves,clocks_white,clocks_black,evals_cp,evals_raw,is_check,is_capture,piece_moved,promotion,is_en_passant,white_rating,black_rating,result,game_end_reason,time_initial,time_increment,utc_timestamp,opening,eco,ply_count,fen
str,str,list[u16],list[u16],list[i16],list[i16],list[bool],list[bool],list[str],list[str],list[bool],i16,i16,str,str,u16,u8,datetime[μs],str,str,u16,str
"""pEz1Av5j""","""e2e4 e7e5 d2d4 e5d4 c2c3 d4c3 …","[600, 589, … 244]","[600, 597, … 281]","[28, 39, … null]","[28, 39, … 32767]","[false, false, … true]","[false, false, … false]","[""p"", ""p"", … ""q""]","["""", """", … """"]","[false, false, … false]",1962,1970,"""1-0""","""mate""",600,0,2017-04-01 00:00:00,"""Danish Gambit""","""C21""",143,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"
"""sUSM4c97""","""e2e4 c7c5 g1f3 e7e6 d2d4 c5d4 …","[600, 589, … 213]","[300, 297, … 79]","[18, 24, … null]","[18, 24, … -32768]","[false, false, … true]","[false, false, … true]","[""p"", ""p"", … ""r""]","["""", """", … """"]","[false, false, … false]",1767,2394,"""0-1""","""mate""",600,0,2017-04-01 00:00:06,"""Sicilian Defense: French Varia…","""B40""",72,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"
"""RhBqMWkT""","""e2e4 e7e5 f1c4 f7f5 c4g8 h8g8 …","[300, 307, … 252]","[300, 306, … 153]","[27, 52, … null]","[27, 52, … -32768]","[false, false, … true]","[false, false, … false]","[""p"", ""p"", … ""q""]","["""", """", … """"]","[false, false, … false]",1777,1752,"""0-1""","""mate""",300,8,2017-04-01 00:00:12,"""Bishop's Opening: Calabrese Co…","""C23""",40,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"
"""SmHjywH8""","""d2d4 d7d5 c2c4 g8f6 e2e3 e7e6 …","[600, 602, … 228]","[600, 600, … 33]","[27, 24, … null]","[27, 24, … 32767]","[false, false, … true]","[false, false, … false]","[""p"", ""p"", … ""q""]","["""", """", … """"]","[false, false, … false]",1651,1534,"""1-0""","""mate""",600,3,2017-04-01 00:00:18,"""Queen's Gambit Refused: Marsha…","""D06""",101,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"
"""mGUJfOtB""","""d2d4 d7d5 c2c4 e7e5 e2e3 c7c5 …","[900, 913, … 476]","[900, 912, … 963]","[12, 16, … 2466]","[12, 16, … 2466]","[false, false, … false]","[false, false, … false]","[""p"", ""p"", … ""r""]","["""", """", … """"]","[false, false, … false]",1656,1781,"""1-0""","""resignation""",900,15,2017-04-01 00:00:38,"""Queen's Gambit Refused: Albin …","""D08""",82,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"


## Stage 2: Tokenized (train/val ready)


In [4]:
pretrain_path = TOKENIZED_DIR / "pretrain"
eval_path = TOKENIZED_DIR / "eval.parquet"

pretrain_dataset = PretrainDataset(pretrain_path)
pretrain_rows = []
for i in range(min(1000, len(pretrain_dataset))):
    tokens, active, opponent, segments, positions = pretrain_dataset[i]
    pretrain_rows.append(
        {
            "token_ids": tokens.tolist(),
            "active_clock_ids": active.tolist(),
            "opponent_clock_ids": opponent.tolist(),
            "segment_ids": segments.tolist(),
            "position_ids": positions.tolist(),
        }
    )
pretrain_sample = pl.DataFrame(pretrain_rows)
eval_sample = pl.read_parquet(eval_path, n_rows=5)

print("Pretrain windows:", len(pretrain_dataset))
print("Eval shape:", pl.read_parquet(eval_path).shape)
print("\nPretrain columns:", pretrain_sample.columns)
print("\nSample packed pretrain windows:")
pretrain_sample

Pretrain windows: 474173
Eval shape: (67219, 3)

Pretrain columns: ['token_ids', 'active_clock_ids', 'opponent_clock_ids', 'segment_ids', 'position_ids']

Sample packed pretrain windows:


token_ids,active_clock_ids,opponent_clock_ids,segment_ids,position_ids
list[i64],list[i64],list[i64],list[i64],list[i64]
"[0, 114, … 6383]","[300, 300, … 265]","[300, 300, … 263]","[0, 0, … 2]","[0, 1, … 44]"
"[0, 114, … 34]","[300, 300, … 194]","[300, 300, … 84]","[0, 0, … 2]","[0, 1, … 88]"
"[0, 111, … 7074]","[300, 300, … 92]","[300, 300, … 152]","[0, 0, … 2]","[0, 1, … 98]"
"[0, 111, … 5380]","[300, 300, … 281]","[300, 300, … 295]","[0, 0, … 3]","[0, 1, … 12]"
"[0, 111, … 6063]","[300, 300, … 335]","[300, 300, … 324]","[0, 0, … 3]","[0, 1, … 12]"
…,…,…,…,…
"[0, 112, … 5325]","[300, 300, … 359]","[300, 300, … 464]","[0, 0, … 2]","[0, 1, … 42]"
"[0, 113, … 5942]","[600, 600, … 179]","[600, 600, … 190]","[0, 0, … 3]","[0, 1, … 78]"
"[0, 114, … 1487]","[300, 300, … 65]","[300, 300, … 326]","[0, 0, … 1]","[0, 1, … 137]"


### Opening analysis


In [5]:
# Load filtered games and analyze openings
raw_lf = pl.scan_parquet(str(FILTERED_DIR / "*.parquet"))

openings = (
    raw_lf.select("opening")
    .collect()["opening"]
    .str.split(":")
    .list.get(0)
    .str.split(",")
    .list.get(0)
    .str.replace(r"#\d+", "")
    .str.strip_chars()
    .str.replace_all(r"\s+", " ")
    .unique()
    .sort()
)

print("unique normalized openings:", len(openings))
for opening in openings:
    print(opening)

unique normalized openings: 168
Alekhine Defense
Amar Opening
Amazon Attack
Amsterdam Attack
Anderssen Opening
Australian Defense
Barnes Defense
Barnes Opening
Benko Gambit
Benko Gambit Accepted
Benko Gambit Declined
Benoni Defense
Bird Opening
Bishop's Opening
Blackmar-Diemer
Blackmar-Diemer Gambit
Blackmar-Diemer Gambit Declined
Blumenfeld Countergambit
Blumenfeld Countergambit Accepted
Boden-Kieseritzky Gambit
Bogo-Indian Defense
Borg Defense
Borg Opening
Bronstein Gambit
Budapest Defense
Canard Opening
Caro-Kann Defense
Carr Defense
Catalan Opening
Center Game
Center Game Accepted
Clemenz Opening
Colle System
Crab Opening
Creepy Crawly Formation
Czech Defense
Danish Gambit
Danish Gambit Accepted
Danish Gambit Declined
Doery Defense
Duras Gambit
Dutch Defense
East Indian Defense
Elephant Gambit
English Defense
English Opening
English Orangutan
English Rat
Englund Gambit
Englund Gambit Complex
Englund Gambit Complex Declined
Englund Gambit Declined
Formation
Four Knights
Four Knights

### Sequence statistics


In [6]:
# length of the longest game in raw data (by move count)
raw_with_lengths = raw_lf.select(
    pl.col("uci_moves").str.split(" ").list.len().alias("move_count"),
    pl.col("uci_moves"),
).collect()

max_length = raw_with_lengths["move_count"].max()
max_idx = raw_with_lengths["move_count"].arg_max()
longest_game_moves = raw_with_lengths[max_idx, "uci_moves"]

print("longest game length (moves):", max_length)
print("example longest game:")
print(longest_game_moves)

longest game length (moves): 600
example longest game:
g2g3 e7e5 f1g2 c7c5 b1c3 d7d6 c3d5 b8c6 c2c4 g8e7 e2e4 c6d4 g1e2 d4e2 d1e2 c8d7 d5e7 f8e7 e1g1 e8g8 a1b1 g8h8 e2d3 d8e8 g1h1 d7c6 f2f3 a7a6 d3e3 f7f6 e3e2 c6d7 e2d1 d7e6 d1c2 e6g8 b2b3 e8f7 c2d1 f7e6 d1e2 a8e8 d2d3 e6c8 c1e3 e7d8 e2d2 d8b6 a2a3 c8d8 d2f2 b6c7 f2e2 d8c8 e2d2 g8e6 f1f2 e6d7 b1b2 d7h3 d2d1 h3d7 g2f1 c7a5 h1g2 a5c7 g2h1 c8d8 f1g2 e8e7 f2f1 b7b6 b2f2 e7e8 f2d2 d8e7 d2a2 d7c6 d1d2 c7b8 d2e1 e7b7 e1f2 b8c7 a3a4 b7a8 f2b2 a8b8 b2a3 b8c8 a2a1 c8b8 a3b4 b8c8 b4c3 c8d7 a1e1 d7e6 e1d1 e8d8 c3c1 d8e8 c1d2 e8d8 d2f2 d8e8 f2g1 e8d8 f1f2 f8f7 g1f1 h8g8 g2h3 d8f8 h3g2 e6e7 f1e2 c6d7 g2f1 d7e6 f2g2 e7d7 g2g1 f7e7 f1g2 f8e8 g1f1 d7d8 f1f2 d8c8 h1g1 c7b8 g1f1 b8c7 f1e1 c8b8 e1d2 b8b7 d2c2 b7c6 d1b1 c6d7 e2e1 d7c8 e1d1 c8b7 e3d2 b7c6 d2c3 e7f7 d1e1 e6d7 g2f1 e8d8 f1e2 d7e8 e2d1 e8d7 e1d2 d7c8 d2c1 c6d7 c2b2 d7c6 b2a3 c8e6 b1b2 c6b7 b2d2 d8e8 a3b2 f7e7 c1c2 b7c8 c2b1 c8d7 d1c2 e6h3 b2c1 h3e6 d2e2 c7b8 c1d2 h7h6 d2e1 b8c7 e1f1 d7c8 f1g1 

In [7]:
# count number of <game_start> and <game_end> tokens in sampled packed pretrain windows
flat_tokens = pl.col("token_ids").explode()
counts = pretrain_sample.select(
    flat_tokens.eq(GAME_START_ID).sum().alias("game_start_count"),
    flat_tokens.eq(GAME_END_ID).sum().alias("game_end_count"),
)

print("Tokenized column checked: token_ids")
print("<game_start> token ID:", GAME_START_ID)
print("<game_end> token ID:", GAME_END_ID)
print("total <game_start> tokens in sample:", int(counts["game_start_count"][0]))
print("total <game_end> tokens in sample:", int(counts["game_end_count"][0]))

Tokenized column checked: token_ids
<game_start> token ID: 0
<game_end> token ID: 1
total <game_start> tokens in sample: 3157
total <game_end> tokens in sample: 2166


### Tokenized game example


In [8]:
example_game = eval_sample.head(1).to_dicts()[0]
# or pick shortest game from eval_sample
# example_game = eval_sample.sort(pl.col("token_ids").list.len()).head(1).to_dicts()[0]
token_ids = example_game["token_ids"]

print("Token IDs:", token_ids)
print("Total tokens:", len(token_ids))

Token IDs: [0, 112, 15, 15, 5380, 82, 34, 1561, 46, 34, 5352, 1534, 5387, 1161, 5416, 549, 5242, 315, 67, 34, 5407, 2355, 4351, 940, 4997, 1633, 67, 34, 5491, 582, 6046, 24, 26, 1407, 4275, 455, 4080, 24, 26, 383, 5577, 1479, 5640, 24, 25, 827, 93, 34, 4763, 1357, 5493, 1628, 5200, 2532, 24, 26, 5435, 379, 7530, 48, 102, 3000, 24, 25, 1]
Total tokens: 65


In [9]:
# Same game decoded as string tokens
tokens_decoded = [ID_TO_MOVE.get(tid, f"<{tid}>") for tid in token_ids]

print("Tokens as strings:")
for i, token in enumerate(tokens_decoded):
    print(f"  {i}: {token}")

print(f"\nTotal tokens: {len(tokens_decoded)}")

Tokens as strings:
  0: <game_start>
  1: <tc_blitz_inc>
  2: <elo_1500_1599>
  3: <elo_1500_1599>
  4: w:pawn:d2d4
  5: <whats_on_h6>
  6: <empty>
  7: b:pawn:e7e5
  8: <whats_on_d2>
  9: <empty>
  10: w:pawn:c2c3
  11: b:pawn:d7d6
  12: w:pawn:d4e5
  13: b:knight:b8c6
  14: w:pawn:e5d6
  15: b:bishop:f8d6
  16: w:knight:g1f3
  17: b:bishop:c8g4
  18: <whats_on_a5>
  19: <empty>
  20: w:pawn:e2e3
  21: b:queen:d8e7
  22: w:bishop:f1e2
  23: b:king:e8c8
  24: w:knight:b1d2
  25: b:pawn:h7h5
  26: <whats_on_a5>
  27: <empty>
  28: w:pawn:h2h3
  29: b:bishop:g4e6
  30: w:queen:d1a4
  31: <is_check>
  32: <no_check>
  33: b:knight:g8f6
  34: w:bishop:e2b5
  35: b:bishop:e6d7
  36: w:bishop:b5c6
  37: <is_check>
  38: <no_check>
  39: b:bishop:d7c6
  40: w:queen:a4a7
  41: b:pawn:b7b6
  42: w:queen:a7a6
  43: <is_check>
  44: <yes_check>
  45: b:king:c8d7
  46: <whats_on_c8>
  47: <empty>
  48: w:king:e1g1
  49: b:knight:f6g4
  50: w:pawn:h3g4
  51: b:pawn:h5g4
  52: w:knight:f3d4
  53: b: